In [1]:
# Cell 1: Execute Multi-Page Pipeline
from pathlib import Path
import pandas as pd
from fb_scraper_core import fetch_latest_10_posts, save_new_posts_to_parquet

# 1. นิยามเพจที่ต้องการดึงทั้งหมด
TARGET_PAGES = {
    "PatumTourist": "100068138844126",
    "rangsitcitypathumthani": "100064659283170"
}

# สร้าง Dictionary เพื่อเก็บตาราง DataFrame ของแต่ละเพจไว้ชั่วคราว
results_dict = {}

# 2. วนลูปสั่งทำงานทีละเพจ
for page_name, page_id in TARGET_PAGES.items():
    print(f"\n{'='*40}")
    print(f"🚀 เริ่มประมวลผลเพจ: {page_name}")
    print(f"{'='*40}")
    
    # สร้าง/ระบุโฟลเดอร์ของเพจนั้นๆ
    SAVE_DIR = Path.home() / "Desktop" / page_name
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    
    # เรียกใช้ฟังก์ชันจากคลังเครื่องมือ (fb_scraper_core.py)
    raw_data = fetch_latest_10_posts(page_id)
    final_df = save_new_posts_to_parquet(raw_data, SAVE_DIR)
    
    # เก็บตารางลงตัวแปรเผื่อเรียกใช้งานทีหลัง
    results_dict[page_name] = final_df
    
    # โชว์ผลลัพธ์เป็นตาราง DataFrame สวยๆ
    if final_df is not None and not final_df.empty:
        print(f"✅ ได้ข้อมูลใหม่ {len(final_df)} แถว:")
        display(final_df) # โชว์ตารางเต็มๆ ใน Cell
    else:
        print(f"✨ ไม่มีข้อมูลใหม่สำหรับเพจ {page_name}")


🚀 เริ่มประมวลผลเพจ: PatumTourist
⏳ กำลังดึง 10 โพสต์ล่าสุดจาก ID: 100068138844126...
🎉 บันทึกข้อมูลใหม่ 3 โพสต์ ลงในไฟล์: fb_posts_2026-03-31.parquet
✅ ได้ข้อมูลใหม่ 3 แถว:


,id,post_date,post_content
0,1264183465862936,2026-03-31 07:45:27,ดอกบัวหลวง สวนเทพปทุม ในสระน้ำหน้าหอคอย สวยจนต...
1,1263450289269587,2026-03-31 02:00:03,🤖🔥 Click Robot Fighting\nเปิดศึกการแข่งขันหุ่น...
2,1264015452546404,2026-03-31 01:57:08,ร้านนี้ของดีตลาดปทุม #ชายขายกล้วย ที่ขายมากว่า...



🚀 เริ่มประมวลผลเพจ: rangsitcitypathumthani
⏳ กำลังดึง 10 โพสต์ล่าสุดจาก ID: 100064659283170...
🎉 บันทึกข้อมูลใหม่ 3 โพสต์ ลงในไฟล์: fb_posts_2026-03-31.parquet
✅ ได้ข้อมูลใหม่ 3 แถว:


,id,post_date,post_content
0,1392647219567241,2026-03-31 05:29:26,ลาเต้ สตูดิโอ รังสิต\nจัดโปรถ่ายภาพแฟชั่น น้อง...
1,1392589709572992,2026-03-31 04:01:05,🤖🔥 Click Robot Fighting\nเปิดศึกการแข่งขันหุ่น...
2,1390909663074330,2026-03-29 04:58:35,ลดสูงสุด 80% มหกรรมฉลอง 20 ปี ฮาร์ดแวร์เฮาส์ ส...


In [5]:
# Cell 2: ตรวจสอบข้อมูลที่ถูกเซฟไปแล้วทั้งหมด (Master DataFrame) พร้อมตัดเนื้อหาซ้ำ
import pandas as pd
from pathlib import Path

TARGET_PAGES = ["PatumTourist", "rangsitcitypathumthani"]
all_data_frames = []

print("📊 สรุปข้อมูลทั้งหมดที่มีอยู่ในไฟล์ Parquet ล่าสุด:\n")

for page_name in TARGET_PAGES:
    folder_path = Path.home() / "Desktop" / page_name
    parquet_files = list(folder_path.glob("*.parquet"))
    
    if parquet_files:
        # โหลดไฟล์ล่าสุดของเพจนั้นๆ มาดู
        parquet_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_file = parquet_files[0]
        
        df = pd.read_parquet(latest_file)
        
        # 🌟 DE Trick: เพิ่มคอลัมน์ชื่อเพจเข้าไป เพื่อให้รู้ว่าแถวไหนมาจากเพจอะไร
        df.insert(0, 'page_name', page_name) 
        all_data_frames.append(df)

# นำตารางของทุกเพจมาต่อกัน
if all_data_frames:
    master_df = pd.concat(all_data_frames, ignore_index=True)
    
    # ==========================================
    # 🌟 เติมส่วนตัดข้อมูลซ้ำตรงนี้ครับ
    # ลบแถวที่เนื้อหา (post_content) ซ้ำกันเป๊ะๆ ออก โดยเก็บอันที่ดึงมาเจออันแรกไว้
    master_df_cleaned = master_df.drop_duplicates(subset=['post_content'], keep='first')
    
    print(f"📊 ข้อมูลรวมทั้งหมด (ก่อนตัด): {len(master_df)} แถว")
    print(f"🧹 ข้อมูลหลังตัดเนื้อหาที่โพสต์ซ้ำกันออก: {len(master_df_cleaned)} แถว\n")
    
    display(master_df_cleaned) # โชว์ตารางที่คลีนแล้ว
    # ==========================================
    
else:
    print("❌ ยังไม่มีไฟล์ข้อมูลในเครื่องเลยครับ")

📊 สรุปข้อมูลทั้งหมดที่มีอยู่ในไฟล์ Parquet ล่าสุด:

📊 ข้อมูลรวมทั้งหมด (ก่อนตัด): 6 แถว
🧹 ข้อมูลหลังตัดเนื้อหาที่โพสต์ซ้ำกันออก: 6 แถว



,page_name,id,post_date,post_content
0,PatumTourist,1264183465862936,2026-03-31 07:45:27,ดอกบัวหลวง สวนเทพปทุม ในสระน้ำหน้าหอคอย สวยจนต...
1,PatumTourist,1263450289269587,2026-03-31 02:00:03,🤖🔥 Click Robot Fighting\nเปิดศึกการแข่งขันหุ่น...
2,PatumTourist,1264015452546404,2026-03-31 01:57:08,ร้านนี้ของดีตลาดปทุม #ชายขายกล้วย ที่ขายมากว่า...
3,rangsitcitypathumthani,1392647219567241,2026-03-31 05:29:26,ลาเต้ สตูดิโอ รังสิต\nจัดโปรถ่ายภาพแฟชั่น น้อง...
4,rangsitcitypathumthani,1392589709572992,2026-03-31 04:01:05,🤖🔥 Click Robot Fighting\nเปิดศึกการแข่งขันหุ่น...
5,rangsitcitypathumthani,1390909663074330,2026-03-29 04:58:35,ลดสูงสุด 80% มหกรรมฉลอง 20 ปี ฮาร์ดแวร์เฮาส์ ส...
